# 🔧 Hands-on: designing the tool surface (Domain 2 — Tools & MCP, 18%)

You'll feel *why* tool design decides agent reliability: prescriptive descriptions,
strict schemas, structured errors the model can recover from, forcing a tool, parallel
calls, and the MCP request shape. Live calls on `claude-haiku-4-5`; a full pass is a few cents.


In [ ]:
# Setup — run me first
%pip install -q anthropic

import os, getpass, json
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key (input is hidden): ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"   # cheap + fast for learning; swap to claude-opus-4-8 for the strongest reasoning

def check(name, fn, hint=""):
    try: ok = fn()
    except Exception as e: ok=False; hint=(hint+f"  (raised {type(e).__name__}: {e})").strip()
    print(f"{'\u2705' if ok else '\u274c'}  {name}" + (f"  \u2014 {hint}" if not ok and hint else ""))
    return ok

print("Client ready \u2713   model =", MODEL)


## Exercise 1 — a vague tool vs a prescriptive one
The model decides *when* to call a tool from its **description**. Watch a thin
description underperform.

**TODO:** rewrite `good_tool`'s description to say *when* to call it ("Call this whenever
the user asks about weather, forecasts, or temperature in a place").


In [ ]:
def get_weather(city: str) -> str:
    return f"18\u00b0C and clear in {city}"

vague_tool = {"name": "get_weather", "description": "weather",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}

# TODO: give good_tool a prescriptive description (when to call it + what it does)
good_tool = {"name": "get_weather", "description": "weather",   # <-- improve this
    "input_schema": {"type": "object",
        "properties": {"city": {"type": "string", "description": "City name, e.g. 'Delhi'"}},
        "required": ["city"], "additionalProperties": False}}

def asks(tool):
    r = client.messages.create(model=MODEL, max_tokens=300, tools=[tool],
        messages=[{"role": "user", "content": "Is it warm in Mumbai right now?"}])
    return r.stop_reason == "tool_use"
print("vague tool triggers:", asks(vague_tool))
print("good  tool triggers:", asks(good_tool))


In [ ]:
check("good_tool description is prescriptive",
      lambda: len(good_tool["description"]) > 30 and "call" in good_tool["description"].lower(),
      "say WHEN to call it, not just 'weather'")


## Exercise 2 — strict tool use guarantees valid params
`strict: True` (with `additionalProperties: false` + `required`) makes the model's
`tool_use.input` validate exactly against your schema — no missing keys, no hallucinated
enum values.

**TODO:** add `"strict": True` and an `enum` of `["celsius", "fahrenheit"]` to `unit`.


In [ ]:
strict_tool = {
    "name": "get_weather",
    "description": "Get current weather. Call this whenever the user asks about weather in a place.",
    # TODO: add  "strict": True,
    "input_schema": {"type": "object",
        "properties": {"city": {"type": "string"},
                       "unit": {"type": "string"}},   # TODO: add "enum": [...]
        "required": ["city", "unit"], "additionalProperties": False}}

r = client.messages.create(model=MODEL, max_tokens=300, tools=[strict_tool],
    tool_choice={"type": "tool", "name": "get_weather"},
    messages=[{"role": "user", "content": "Weather in Delhi, in Fahrenheit?"}])
call = next((b for b in r.content if b.type == "tool_use"), None)
print(call.input if call else "(no call)")


In [ ]:
check("tool is strict", lambda: strict_tool.get("strict") is True, "add strict: True")
check("unit is an enum",
      lambda: set(strict_tool["input_schema"]["properties"]["unit"]["enum"]) == {"celsius", "fahrenheit"},
      "add enum: ['celsius','fahrenheit']")
check("model produced a valid unit", lambda: call.input["unit"] in ("celsius", "fahrenheit"))


## Exercise 3 — structured errors let the model recover
When a tool fails, don't crash — return a `tool_result` with `is_error: True` and an
*actionable* message. The model reads it and adapts.

**TODO:** build the error `tool_result` block (`is_error: True`, a helpful `content`).


In [ ]:
lookup_tool = {"name": "lookup_user", "description": "Look up a user by exact id.",
    "input_schema": {"type": "object", "properties": {"user_id": {"type": "string"}},
                     "required": ["user_id"], "additionalProperties": False}}

msgs = [{"role": "user", "content": "Look up user 'ada'. If not found, ask me for the numeric id."}]
r1 = client.messages.create(model=MODEL, max_tokens=400, tools=[lookup_tool], messages=msgs)
msgs.append({"role": "assistant", "content": r1.content})
call = next((b for b in r1.content if b.type == "tool_use"), None)

# The real DB has no 'ada' — return a STRUCTURED error, not a crash.
# TODO: make err_block a tool_result with is_error True and a message telling the model what to do.
err_block = {"type": "tool_result", "tool_use_id": call.id, "content": ""}   # <-- fix
msgs.append({"role": "user", "content": [err_block]})
r2 = client.messages.create(model=MODEL, max_tokens=400, tools=[lookup_tool], messages=msgs)
reply = next((b.text for b in r2.content if b.type == "text"), "")
print(reply)


In [ ]:
check("error block is flagged", lambda: err_block.get("is_error") is True, "set is_error: True")
check("model recovered gracefully (asked for id / acknowledged)",
      lambda: any(w in reply.lower() for w in ("id", "not found", "couldn", "provide")),
      "a good error message steers the model's next move")


## Exercise 4 — parallel tool calls
One assistant turn can contain **several** `tool_use` blocks. Run them all, then return
**all** `tool_result` blocks in a **single** user message (splitting them trains the model
to stop parallelizing).

**TODO:** collect one `tool_result` per `tool_use` block into `results`.


In [ ]:
def temp(city): return f"{{'{city}': '20\u00b0C'}}"
wtool = {"name": "get_weather", "description": "Weather for one city. Call once per city.",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}},
                     "required": ["city"], "additionalProperties": False}}
msgs = [{"role": "user", "content": "Compare the weather in Delhi and Mumbai. Call the tool for each."}]
r = client.messages.create(model=MODEL, max_tokens=500, tools=[wtool], messages=msgs)
calls = [b for b in r.content if b.type == "tool_use"]
print("parallel calls in one turn:", len(calls))

results = []
for b in calls:
    out = temp(b.input["city"])
    # TODO: append a tool_result (type, tool_use_id=b.id, content=out) to results
    pass


In [ ]:
check("one result per parallel call", lambda: len(results) == len(calls) and len(calls) >= 1)
check("each result carries a matching tool_use_id",
      lambda: all(r_.get("tool_use_id") for r_ in results) and
              {r_["tool_use_id"] for r_ in results} == {b.id for b in calls})


## 🌶️ Stretch — the MCP connector request shape
MCP servers expose third-party tools. On the Messages API you declare the server in
`mcp_servers` **and** reference it with an `mcp_toolset` — both halves are required, or
the request 400s.

**TODO:** build `mcp_servers` and `tools` so the toolset's `mcp_server_name` matches the
server's `name`. (This is a shape check — no live call, since it needs a real server.)


In [ ]:
SERVER_URL = "https://mcp.example.com/sse"
# TODO: give the server a name and reference it from the toolset
mcp_servers = [{"type": "url", "url": SERVER_URL, "name": ""}]        # <-- name it
tools = [{"type": "mcp_toolset", "mcp_server_name": ""}]              # <-- same name


In [ ]:
check("server is named", lambda: bool(mcp_servers[0]["name"]))
check("toolset references the same server",
      lambda: tools[0]["mcp_server_name"] == mcp_servers[0]["name"],
      "mcp_toolset.mcp_server_name must equal a server's name")


---
**Domain-2 takeaways (exam):** the tool *description* is the routing signal; `strict`
guarantees valid inputs; `is_error` results keep the loop alive; parallel results go back
in one message; MCP needs both `mcp_servers` and a matching `mcp_toolset`.
